In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [3]:
train_df = pd.read_csv("../data/train.csv")
train_df = train_df.drop("event_id", axis=1)


test_df = pd.read_csv("../data/test.csv")
test_event_id = test_df["event_id"].copy()

meta_df = pd.read_csv("../data/metaData.csv")

submission = pd.read_csv("../data/sample_submission.csv")

In [ ]:
train_df.head() 


,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,log_area_ratio_0_5h,...,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,3,4.265188,0,79.696304,2.875935,0.036086,0.674281,4.390693,1.354787,0.03545,...,0.886373,-0.054649,0.054649,-1.937219,-0.106026,19,4,5,18.892512,0
1,2,1.169918,0,8.946749,0.000000,0.000000,0.000000,2.297246,0.000000,0.00000,...,0.000000,-0.568898,0.568898,-0.000000,-0.000000,4,4,6,22.048108,1
2,4,4.777526,0,106.482638,0.000000,0.000000,0.000000,4.677329,0.000000,0.00000,...,0.000000,0.882385,0.882385,0.000000,0.000000,22,4,8,0.888895,1
3,1,0.000000,1,67.631125,0.000000,0.000000,0.000000,4.228746,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,20,5,8,60.953021,0
4,2,4.975273,0,35.632874,0.000000,0.000000,0.000000,3.600946,0.000000,0.00000,...,0.000000,0.934634,0.934634,-0.000000,0.000000,21,5,7,44.990274,0


In [5]:
print(f"Number of training samples: {train_df.shape[0]}")
print(f"Number of features: {train_df.shape[1]}")
print(f"Number of test samples: {test_df.shape[0]}")

Number of training samples: 221
Number of features: 36
Number of test samples: 95


In [8]:
n_hits = int(train_df['event'].sum())
n_censored = len(train_df) - n_hits
hit_times = train_df.loc[train_df['event'] == 1, 'time_to_hit_hours']


print("DATASET SUMMARY")

print(f"Training set:   {train_df.shape[0]} fires with {train_df.shape[1]} columns")
print(f"Test set:       {test_df.shape[0]} fires with {test_df.shape[1]} columns")
print()
print("Target distribution in training data:")
print(f"  Fires that HIT:        {n_hits} ({n_hits/len(train_df)*100:.1f}%)")
print(f"  Censored (no hit):     {n_censored} ({n_censored/len(train_df)*100:.1f}%)")
print()
print("Time to hit (for fires that hit):")
print(f"  Average: {hit_times.mean():.1f} hours")
print(f"  Median:  {hit_times.median():.1f} hours")
print(f"  Fastest: {hit_times.min():.2f} hours")
print(f"  Slowest: {hit_times.max():.1f} hours")
print()
print(f"Missing values: {train_df.isnull().sum().sum()}")

DATASET SUMMARY
Training set:   221 fires with 36 columns
Test set:       95 fires with 35 columns

Target distribution in training data:
  Fires that HIT:        69 (31.2%)
  Censored (no hit):     152 (68.8%)

Time to hit (for fires that hit):
  Average: 10.0 hours
  Median:  3.5 hours
  Fastest: 0.00 hours
  Slowest: 66.9 hours

Missing values: 0


In [9]:
train_df.info()
train_df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 221 entries, 0 to 220
Data columns (total 36 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   num_perimeters_0_5h           221 non-null    int64  
 1   dt_first_last_0_5h            221 non-null    float64
 2   low_temporal_resolution_0_5h  221 non-null    int64  
 3   area_first_ha                 221 non-null    float64
 4   area_growth_abs_0_5h          221 non-null    float64
 5   area_growth_rel_0_5h          221 non-null    float64
 6   area_growth_rate_ha_per_h     221 non-null    float64
 7   log1p_area_first              221 non-null    float64
 8   log1p_growth                  221 non-null    float64
 9   log_area_ratio_0_5h           221 non-null    float64
 10  relative_growth_0_5h          221 non-null    float64
 11  radial_growth_m               221 non-null    float64
 12  radial_growth_rate_m_per_h    221 non-null    float64
 13  centr

,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,log_area_ratio_0_5h,...,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
count,221.000000,221.000000,221.000000,221.000000,221.000000,2.210000e+02,221.000000,221.000000,221.000000,2.210000e+02,...,221.000000,221.000000,221.000000,221.000000,221.000000,221.000000,221.000000,221.000000,221.000000,221.000000
mean,2.063348,0.979869,0.728507,619.131641,26.332398,1.789087e-01,6.167128,4.683276,0.389346,6.543391e-02,...,0.046000,-0.004971,0.172704,1.617188,0.551690,15.429864,2.841629,6.782805,37.567626,0.312217
std,2.578859,1.738052,0.445739,1447.723668,187.437018,1.302001e+00,40.467370,2.083529,1.340348,3.003211e-01,...,0.171690,0.371909,0.329210,37.789199,46.760648,7.921250,1.974217,1.571876,25.902361,0.464450
min,1.000000,0.000000,0.000000,0.037525,-0.000022,-1.437844e-07,-0.000005,0.036838,0.000000,-1.437844e-07,...,0.000000,-0.999995,0.000000,-213.411731,-526.597241,0.000000,0.000000,1.000000,0.001220,0.000000
25%,1.000000,0.000000,0.000000,25.219058,0.000000,0.000000e+00,0.000000,3.266487,0.000000,0.000000e+00,...,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,1.000000,6.000000,12.242322,0.000000
50%,1.000000,0.000000,1.000000,110.149250,0.000000,0.000000e+00,0.000000,4.710874,0.000000,0.000000e+00,...,0.000000,0.000000,0.000000,0.000000,0.000000,19.000000,3.000000,7.000000,43.109830,0.000000
75%,2.000000,1.356107,1.000000,485.092561,0.000000,0.000000e+00,0.000000,6.186399,0.000000,0.000000e+00,...,0.000000,0.000000,0.071697,0.000000,0.000000,21.000000,5.000000,8.000000,63.938706,1.000000
max,17.000000,4.994457,1.000000,11942.392115,2508.041442,1.788970e+01,520.443033,9.387933,7.827656,2.938617e+00,...,0.917415,0.994594,0.999995,277.110446,383.099186,23.000000,6.000000,9.000000,66.994474,1.000000


In [10]:
X = train_df.drop(["time_to_hit_hours", "event"], axis=1)
y = train_df[["event", "time_to_hit_hours"]]

X_test = test_df.copy()


# try this later to see if performance improves 
#one-hot encode the event_start_month column and drop the original column
X = pd.get_dummies(X, columns=["event_start_month"], drop_first=True)
#get rid of potentially redundant features
X = X.drop(["area_first_ha", "area_growth_abs_0_5h"], axis=1)

In [11]:
#get all features
feature_cols= [col for col in train_df.columns if col not in ["event","time_to_hit_hours"]]
print("Features:", feature_cols)

#caclculte correlation with target
corr_with_event = train_df[feature_cols].corrwith(train_df['event']).abs()
corr_with_time = train_df[feature_cols].corrwith(train_df['time_to_hit_hours']).abs()

print("Top features correlated with event:")
print(corr_with_event.sort_values(ascending=False).head(10) )
print()
print("Top features correlated with time to hit:")
print(corr_with_time.sort_values(ascending=False).head(10) )

Features: ['num_perimeters_0_5h', 'dt_first_last_0_5h', 'low_temporal_resolution_0_5h', 'area_first_ha', 'area_growth_abs_0_5h', 'area_growth_rel_0_5h', 'area_growth_rate_ha_per_h', 'log1p_area_first', 'log1p_growth', 'log_area_ratio_0_5h', 'relative_growth_0_5h', 'radial_growth_m', 'radial_growth_rate_m_per_h', 'centroid_displacement_m', 'centroid_speed_m_per_h', 'spread_bearing_deg', 'spread_bearing_sin', 'spread_bearing_cos', 'dist_min_ci_0_5h', 'dist_std_ci_0_5h', 'dist_change_ci_0_5h', 'dist_slope_ci_0_5h', 'closing_speed_m_per_h', 'closing_speed_abs_m_per_h', 'projected_advance_m', 'dist_accel_m_per_h2', 'dist_fit_r2_0_5h', 'alignment_cos', 'alignment_abs', 'cross_track_component', 'along_track_speed', 'event_start_hour', 'event_start_dayofweek', 'event_start_month']
Top features correlated with event:
dist_min_ci_0_5h                0.481379
low_temporal_resolution_0_5h    0.379117
num_perimeters_0_5h             0.370501
dt_first_last_0_5h              0.352954
alignment_abs   

In [12]:
train_df.to_csv("../data/train_processed.csv", index=False)
test_df.to_csv("../data/test_processed.csv", index=False)